# DEAI-opdracht: Lineaire Regressie met AmesHousing

In deze opdracht maak ik een model dat de **verkoopprijs (`SalePrice`)** van huizen voorspelt.

Ik gebruik een **lineair regressiemodel met `SGDRegressor`**, omdat ik in deze opdracht wil experimenteren met:
- **epochs** (`max_iter`)
- **learning rate** (`eta0`)

De uitwerking is bewust eenvoudig gehouden, zodat elke stap goed te volgen is.

## Stap 1 en 2: bestand inlezen

We lezen het tabblad **AmesHousing** in als DataFrame.
Daarna bekijken we kort de eerste rijen van de dataset.

In [9]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

In [10]:
bestand = "AmesHousing.xlsx"

df = pd.read_excel(bestand, sheet_name="AmesHousing")
data_dictionary = pd.read_excel(bestand, sheet_name="Data Dictionary")

print("Vorm van de dataset:", df.shape)
display(df.head())
display(data_dictionary.head(10))

Vorm van de dataset: (2930, 12)


,ID,SalePrice,Garage,Overall Qual,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,Neighborhood,House Style
0,1,215000,yes,6,1656,1080.0,31770,1960,1,3,NAmes,1Story
1,2,105000,yes,5,896,882.0,11622,1961,1,2,NAmes,1Story
2,3,172000,yes,6,1329,1329.0,14267,1958,1,3,NAmes,1Story
3,4,244000,yes,7,2110,2110.0,11160,1968,2,3,NAmes,1Story
4,5,189900,yes,5,1629,928.0,13830,1997,2,3,Gilbert,2Story


,Variabele,Betekenis
0,ID,"Uniek nummer per huis, te vergelijken met een ..."
1,SalePrice,Verkoopprijs van het huis (in dollars: USD)
2,Garage,Geeft weer of het huis wel/geen garage bevat
3,Overall Qual,Algemene kwaliteit van materialen en afwerking...
4,Gr Liv Area,Woonoppervlak boven de grond (square feet)
5,Total Bsmt SF,Totale oppervlakte van de kelder
6,Lot Area,Grootte van het perceel (square feet)
7,Year Built,Bouwjaar van het huis
8,Full Bath,Aantal volledige badkamers
9,Bedroom AbvGr,Aantal slaapkamers boven de grond


In [11]:
print("Aantal rijen en kolommen:", df.shape)
print("\nKolommen:")
print(df.columns.tolist())

Aantal rijen en kolommen: (2930, 12)

Kolommen:
['ID', 'SalePrice', 'Garage', 'Overall Qual', 'Gr Liv Area', 'Total Bsmt SF', 'Lot Area', 'Year Built', 'Full Bath', 'Bedroom AbvGr', 'Neighborhood', 'House Style']


## Stap 3: target en top 3 features kiezen

Uit het tabblad **Data Dictionary** blijkt:

- **Targetvariabele:** `SalePrice`
- Betekenis: verkoopprijs van het huis in dollars

Mijn eerste 3 gekozen features zijn:
1. `Overall Qual`
2. `Gr Liv Area`
3. `Neighborhood`

### Waarom deze 3?
- **Overall Qual** zegt iets over de algemene kwaliteit van het huis.
- **Gr Liv Area** zegt iets over de woonoppervlakte.
- **Neighborhood** is een categorische variabele en locatie heeft vaak veel invloed op de verkoopprijs.

## Stap 4: data voorbereiden

### Horizontale split
- `X` = de features
- `y` = de target (`SalePrice`)

### Verticale split
Daarna splits ik de data in:
- `X_train`
- `X_test`
- `y_train`
- `y_test`

Ik gebruik **80% training** en **20% testdata**.

In [12]:
target = "SalePrice"
eerste_features = ["Overall Qual", "Gr Liv Area", "Neighborhood"]

data = df[eerste_features + [target]].dropna(subset=[target]).copy()

X = data[eerste_features]
y = data[target]

print("Horizontale split:")
print("X =", X.shape)
print("y =", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("\nVerticale split:")
print("X_train =", X_train.shape)
print("X_test  =", X_test.shape)
print("y_train =", y_train.shape)
print("y_test  =", y_test.shape)

Horizontale split:
X = (2930, 3)
y = (2930,)

Verticale split:
X_train = (2344, 3)
X_test  = (586, 3)
y_train = (2344,)
y_test  = (586,)


## Stap 5: model maken

Ik gebruik `SGDRegressor` als regressiemodel.

Belangrijke hyperparameters die ik met `help(SGDRegressor)` kan bekijken zijn bijvoorbeeld:
- `max_iter` = aantal epochs
- `eta0` = learning rate
- `learning_rate`
- `penalty`

In deze opdracht focus ik vooral op:
- `max_iter`
- `eta0`

In [13]:
# Deze regel kun je gebruiken om alle hyperparameters te bekijken:
# help(SGDRegressor)

print("Hyperparameters die ik in dit notebook gebruik:")
print("- max_iter")
print("- eta0")

Hyperparameters die ik in dit notebook gebruik:
- max_iter
- eta0


## Handige functie voor experimenten

Deze functie:
1. kiest de features
2. vult missende waarden op
3. voert one-hot encoding uit voor categorische kolommen
4. splitst de data in train en test
5. traint het model
6. berekent de evaluatiemetrics

In [14]:
def voer_experiment_uit(features, max_iter=1000, eta0=0.0001, run_naam="Run"):
    data = df[features + ["SalePrice"]].dropna(subset=["SalePrice"]).copy()

    X = data[features]
    y = data["SalePrice"]

    categorische_kolommen = X.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns.tolist()
    numerieke_kolommen = [kolom for kolom in features if kolom not in categorische_kolommen]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numerieke_kolommen
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]),
                categorische_kolommen
            )
        ]
    )

    model = SGDRegressor(
        max_iter=max_iter,
        eta0=eta0,
        learning_rate="constant",
        random_state=42
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    pipeline.fit(X_train, y_train)
    voorspellingen = pipeline.predict(X_test)

    resultaat = {
        "Run": run_naam,
        "Features": ", ".join(features),
        "Aantal features": len(features),
        "Categorische features": ", ".join(categorische_kolommen) if categorische_kolommen else "-",
        "Epochs (max_iter)": max_iter,
        "Learning rate (eta0)": eta0,
        "MAE": mean_absolute_error(y_test, voorspellingen),
        "RMSE": np.sqrt(mean_squared_error(y_test, voorspellingen)),
        "R2": r2_score(y_test, voorspellingen),
        "MAPE": mean_absolute_percentage_error(y_test, voorspellingen),
        "X_train_shape": X_train.shape,
        "X_test_shape": X_test.shape,
        "y_train_shape": y_train.shape,
        "y_test_shape": y_test.shape
    }

    return resultaat

## Stap 6 en 7: initiële run en experimenten

Ik begin met een **initiële run** op basis van 3 logische features.

Daarna voer ik meerdere experimenten uit:
- eerst met extra features
- daarna met aangepaste hyperparameters

Zo kan ik vergelijken welke combinatie het beste werkt.

In [15]:
resultaten = []

resultaten.append(voer_experiment_uit(
    features=["Overall Qual", "Gr Liv Area", "Neighborhood"],
    max_iter=1000,
    eta0=0.0001,
    run_naam="Initiële run"
))

resultaten.append(voer_experiment_uit(
    features=["Overall Qual", "Gr Liv Area", "Neighborhood", "Year Built"],
    max_iter=1000,
    eta0=0.0001,
    run_naam="Experiment 1"
))

resultaten.append(voer_experiment_uit(
    features=["Overall Qual", "Gr Liv Area", "Neighborhood", "Year Built", "House Style"],
    max_iter=1000,
    eta0=0.0001,
    run_naam="Experiment 2"
))

resultaten.append(voer_experiment_uit(
    features=["Overall Qual", "Gr Liv Area", "Neighborhood", "Year Built", "House Style", "Total Bsmt SF"],
    max_iter=1000,
    eta0=0.0001,
    run_naam="Experiment 3"
))

resultaten.append(voer_experiment_uit(
    features=["Overall Qual", "Gr Liv Area", "Neighborhood", "Year Built", "House Style", "Total Bsmt SF"],
    max_iter=3000,
    eta0=0.0003,
    run_naam="Experiment 4"
))

resultaten_df = pd.DataFrame(resultaten)
resultaten_df["MAE"] = resultaten_df["MAE"].round(2)
resultaten_df["RMSE"] = resultaten_df["RMSE"].round(2)
resultaten_df["R2"] = resultaten_df["R2"].round(4)
resultaten_df["MAPE"] = (resultaten_df["MAPE"] * 100).round(2)

display(resultaten_df.sort_values(by="R2", ascending=False))

,Run,Features,Aantal features,Categorische features,Epochs (max_iter),Learning rate (eta0),MAE,RMSE,R2,MAPE,X_train_shape,X_test_shape,y_train_shape,y_test_shape
3,Experiment 3,"Overall Qual, Gr Liv Area, Neighborhood, Year ...",6,"Neighborhood, House Style",1000,0.0001,22684.35,36486.82,0.8340,12.41,"(2344, 6)","(586, 6)","(2344,)","(586,)"
4,Experiment 4,"Overall Qual, Gr Liv Area, Neighborhood, Year ...",6,"Neighborhood, House Style",3000,0.0003,23240.83,36921.83,0.8300,12.90,"(2344, 6)","(586, 6)","(2344,)","(586,)"
2,Experiment 2,"Overall Qual, Gr Liv Area, Neighborhood, Year ...",5,"Neighborhood, House Style",1000,0.0001,23056.70,36969.63,0.8295,12.53,"(2344, 5)","(586, 5)","(2344,)","(586,)"
1,Experiment 1,"Overall Qual, Gr Liv Area, Neighborhood, Year ...",4,Neighborhood,1000,0.0001,25173.43,39419.98,0.8062,13.71,"(2344, 4)","(586, 4)","(2344,)","(586,)"
0,Initiële run,"Overall Qual, Gr Liv Area, Neighborhood",3,Neighborhood,1000,0.0001,25537.47,39671.48,0.8037,14.00,"(2344, 3)","(586, 3)","(2344,)","(586,)"


## Beste experiment kiezen

Bij regressie geldt meestal:
- een lagere **MAE** is beter
- een lagere **RMSE** is beter
- een hogere **R²** is beter
- een lagere **MAPE** is beter

In deze uitwerking kies ik de beste run vooral op basis van **R²**, en ik controleer daarbij ook of de foutmaten logisch blijven.

In [16]:
beste_run = resultaten_df.sort_values(by="R2", ascending=False).iloc[0]

print("Beste run op basis van R²:")
display(beste_run)

Beste run op basis van R²:


Run                                                           Experiment 3
Features                 Overall Qual, Gr Liv Area, Neighborhood, Year ...
Aantal features                                                          6
Categorische features                            Neighborhood, House Style
Epochs (max_iter)                                                     1000
Learning rate (eta0)                                                0.0001
MAE                                                               22684.35
RMSE                                                              36486.82
R2                                                                   0.834
MAPE                                                                 12.41
X_train_shape                                                    (2344, 6)
X_test_shape                                                      (586, 6)
y_train_shape                                                      (2344,)
y_test_shape             

## Stap 8: verantwoording voor portfolio

- De **initiële run** gebruikte 3 logische features: `Overall Qual`, `Gr Liv Area` en `Neighborhood`.
- In **Experiment 1** heb ik `Year Built` toegevoegd.
- In **Experiment 2** heb ik ook `House Style` toegevoegd.
- In **Experiment 3** heb ik daarnaast `Total Bsmt SF` toegevoegd.
- In **Experiment 4** heb ik dezelfde features gebruikt, maar ook de hyperparameters aangepast door meer epochs en een hogere learning rate te kiezen.

### Conclusie
De beste run kies ik op basis van de combinatie van:
- een zo hoog mogelijke **R²**
- een zo laag mogelijke **MAE**
- een zo laag mogelijke **RMSE**
- een zo laag mogelijke **MAPE**

Hieruit blijkt of extra features of aangepaste hyperparameters het meeste effect hebben op de prestaties van het model.